# TISER conditional retention and replay study

This notebook invokes the repository CLIs. It runs C0/C1 retention first, trains
C1R and R25 only after clear forgetting, and runs token sensitivity only above
the predeclared 10% threshold. It contains no ablation or commit-management step.

The project is already synchronized to Google Drive. Select a single-GPU runtime
and run the cells in order. Inputs, adapters, audit state, checkpoints, and results
are read from or written directly to the synchronized project directory. Complete
the semantic audit in that directory before running the final-campaign cells. The
original 113 inputs are never used by smoke tests or model selection.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension')
STUDY_DIR = PROJECT_ROOT / 'results/forgetting_replay/study_v2'
required = [PROJECT_ROOT / 'scripts/experiment.py', PROJECT_ROOT / 'requirements-experiments.txt']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Synchronized project is missing required files: {missing}')
STUDY_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Study outputs:', STUDY_DIR)


Mounted at /content/drive
Project root: /content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension
Study outputs: /content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension/results/forgetting_replay/study_v2


## Install and verify the runtime
Use the runtime's CUDA-matched PyTorch. Exact resolved package versions are recorded and checked on resume.

In [ ]:
%pip install -r requirements.txt -r requirements-experiments.txt

%pip install -e .

import torch

assert torch.cuda.is_available() and torch.cuda.device_count() == 1, 'Use one CUDA GPU.'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Prepare the frozen study and original-TISER populations

In [ ]:
import os

os.chdir(PROJECT_ROOT)

if not (STUDY_DIR / "protocol.json").exists():
    !python scripts/tennis/fetch_retention_data.py

    !python scripts/experiment.py init --study-dir "{STUDY_DIR}"

if not (STUDY_DIR / "data/split_summary.json").exists():
    !python scripts/experiment.py prepare-data --study-dir "{STUDY_DIR}"

print(json.loads((STUDY_DIR / "data/split_summary.json").read_text()))


Verified original TISER revision 7bdac51ea363a71b1805972b1d2c025f5cd173a4
{
  "schema_version": 2,
  "created_at": "2026-09-11T17:42:14.279427+00:00",
  "protocol_sha256": "125aad3ba4d9a10cae71d67c1436a388447fddb11c2bdf881d07d8cc0f090e12",
  "source_sha256": "c5f37f650d4ff388d42ca62ce854969b88470ee6d3fa74b70de962791e22206a",
  "conditions": {
    "C0": {
      "adapter": {
        "scope": "workspace",
        "path": "model/tiser_qwen7b_full/adapter"
      },
      "sha256": "5e35aa65eec27b7e65a7b64f0e5ecedd014459a11cd056a1a1fa55cf01ea612f"
    },
    "C1": {
      "adapter": {
        "scope": "workspace",
        "path": "model/tennis_from_tiser_e2_lr0.0002_bs4_ga4_r16_a32_d0p05_20260616_104036_011/adapter"
      },
      "sha256": "4d1ec1b8f6711b3f935e4f49a6d4635b76fc05ee979ba76d443ae7483342cafe"
    }
  },
  "artifacts": {},
  "status": "prepared; inference and training pending"
}
{
  "n_input": 22014,
  "n_output": 600,
  "sampling_mode": "per_split",
  "split_summary": {
    "te

## Training-data smoke evaluation
The smoke input is five historical training rows. It cannot consume the 113 holdout.
It checks the 7B adapter/loading/generation path before the retention gate.


In [ ]:
from src.experiment.artifacts import freeze
from src.experiment.study import model_revision, save_config, C0
from src.utils.config import load_config

smoke_input = STUDY_DIR / 'smoke/training_inputs.json'

rows = json.loads(
    (PROJECT_ROOT / 'data/tennis/tennis_train_traced_full.json').read_text()
)[:5]

freeze(smoke_input, rows)

cfg = load_config('config/config_tennis_7b_reported_best.yaml')

cfg.paths.test_file = str(smoke_input)
cfg.model.revision = model_revision(STUDY_DIR)
cfg.model.tokenizer_revision = cfg.model.revision
cfg.eval.batch_size = 4

smoke_cfg = save_config(STUDY_DIR / 'smoke/config.yaml', cfg)

!python scripts/tennis/evaluate_tennis.py \
    --config "{smoke_cfg}" \
    --condition training_smoke \
    --adapter-dir "{PROJECT_ROOT / C0}" \
    --output-dir "{STUDY_DIR / 'smoke/evaluation'}" \
    --resume

2026-09-11 17:42:30.152211: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-11 17:42:30.224291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 7.30kB [00:00, 25.0MB/s]
vocab.json: 2.78MB [00:00, 125MB/s]
merges.txt: 1.67MB [00:00, 140MB/s]
tokenizer.json: 7.03MB [00:00, 190MB/s]
config.json: 100% 663/663 [00:00<00:00, 6.42MB/s]
model.safetensors.index.json: 27.8kB [00:00, 121MB/s]
model-00001-of-00004.safetensors:   0% 0.00/3.95G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 10.5M

In [ ]:
# lets the CUDA allocator grow memory segments more flexibly to reduce fragmentation when generation batches vary greatly in length
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Measure forgetting before training replay

In [ ]:
gate_path = STUDY_DIR / "forgetting_gate.json"

if gate_path.exists():
    gate = json.loads(gate_path.read_text())
    print("Using existing forgetting gate.")
else:
    for condition in ["C0", "C1"]:
        !python scripts/experiment.py evaluate \
            --study-dir "{STUDY_DIR}" \
            --condition "{condition}" \
            --domain tiser \
            --stage selection \
            --resume

    !python scripts/experiment.py gate \
        --study-dir "{STUDY_DIR}"

    gate = json.loads(gate_path.read_text())

print(json.dumps(gate, indent=2))


2026-09-11 20:14:06.186127: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-11 20:14:06.256978: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[eval] WARNING: dataset_name values not in config splits: ['tot_semantic_test']
Loading checkpoint shards: 100% 4/4 [00:09<00:00,  2.26s/it]
generate: 100% 1/1 [00:24<00:00, 24.52s/it]
generate: 100% 1/1 [00:26<00:00, 26.07s/it]
generate: 100% 1/1 [00:25<00:00, 25.02s/it]
generate: 100% 1/1 [00:32<00:00, 32.82s/it]
generate: 100% 1/1 [00:19<00:00, 19.98s/it]
generate: 100

## Conditional current control and replay
Checkpoints include optimizer, scheduler, RNG state and token counters. Completed
conditions are reused. An interruption resumes the latest complete checkpoint.
With no complete checkpoint, retain the failed directory and restart from C0
using the CLI's explicit restart-incomplete option.


In [ ]:
def train_if_needed(condition):

    registry = json.loads((STUDY_DIR / 'registry.json').read_text())

    if condition in registry['conditions']:
        print('Already complete:', condition)
        return

    checkpoints = sorted(
        (STUDY_DIR / 'training' / condition / 'trainer').glob('checkpoint-*'),
        key=lambda p: int(p.name.split('-')[-1])
    )

    required = [
        'optimizer.pt',
        'scheduler.pt',
        'rng_state.pth',
        'trainer_state.json',
        'token_exposure.json',
        'training_spec.json'
    ]

    checkpoints = [
        p for p in checkpoints
        if all((p / f).is_file() for f in required)
        and any(
            (p / f).is_file()
            for f in ['adapter_model.safetensors', 'adapter_model.bin']
        )
    ]

    if checkpoints:
        checkpoint = checkpoints[-1]

        !python scripts/experiment.py train \
            --study-dir "{STUDY_DIR}" \
            --condition "{condition}" \
            --resume-from-checkpoint "{checkpoint}"

    elif (
        (STUDY_DIR / 'training' / condition).exists()
        or (STUDY_DIR / 'models' / condition).exists()
    ):
        !python scripts/experiment.py train \
            --study-dir "{STUDY_DIR}" \
            --condition "{condition}" \
            --restart-incomplete

    else:
        !python scripts/experiment.py train \
            --study-dir "{STUDY_DIR}" \
            --condition "{condition}"


if gate['decision'] == 'clear_forgetting':

    !python scripts/experiment.py replay-data \
        --study-dir "{STUDY_DIR}"

    train_if_needed('C1R')

    train_if_needed('R25')

    !python scripts/experiment.py token-gate \
        --study-dir "{STUDY_DIR}"

    tokens = json.loads((STUDY_DIR / 'token_gate.json').read_text())

    print(tokens)

    if tokens['sensitivity_required']:
        train_if_needed('R25-T')

else:
    print('Replay stopped under the frozen rule:', gate['decision'])

Replay stopped under the frozen rule: inconclusive


## Complete and freeze the tennis semantic audit
Import the individually judged semantic A/B responses into this synchronized
project, prepare and import adjudications, and run `scripts/audit.py summarize`
followed by `freeze-views`.
The reflection and trace audits are independent and do not block this experiment.
No predictions are shown to the semantic auditors. This cell deliberately stops
while that audit is pending; it never substitutes partial coverage for completion.


In [ ]:
AUDIT_DIR = PROJECT_ROOT / 'results/tennis_semantic_audit_v2'

!python scripts/audit.py summarize \
    --output-dir "{AUDIT_DIR}"

audit = json.loads((AUDIT_DIR / 'summary.json').read_text())

assert audit['status'] == 'complete', \
    'Complete all primary judgments and adjudications before final evaluation.'

!python scripts/audit.py freeze-views \
    --output-dir "{AUDIT_DIR}"

{
  "version": "offline-audit-v2",
  "status": "complete",
  "expected": {
    "semantic": 1121
  },
  "completed": {
    "semantic": 1121
  },
  "pending": {},
  "two_pass_disagreements_unique_items": {
    "semantic": 9
  },
  "human_calibration": false,
  "actual_snapshot": null,
  "interpretation": "UI-mediated model judgments; shared judge bias and accuracy remain uncalibrated.",
  "coverage": {
    "semantic/dev/all/duration_minutes": {
      "supported": 15
    },
    "semantic/dev/all/immediate_before_after": {
      "supported": 20
    },
    "semantic/dev/all/other_temporal": {
      "supported": 2
    },
    "semantic/dev/all/overlap_while_during": {
      "supported": 11
    },
    "semantic/dev/all/tennis_injury_or_medical": {
      "supported": 1
    },
    "semantic/dev/all/tournament_round_sequence": {
      "supported": 2
    },
    "semantic/dev/all/which_first_last": {
      "supported": 23
    },
    "semantic/dev/all/yes_no_before_after": {
      "supported": 39
  

## Final tennis holdout evaluation

Evaluate C0 and C1 on the frozen 113-record tennis holdout. The retention
comparison and forgetting decision were already calculated from the predefined
TISER sample. This cell does not evaluate the full TISER complement and does
not affect training or model selection.

In [ ]:
campaign_path = STUDY_DIR / "final_campaign.json"

if not campaign_path.exists():
    registry = json.loads((STUDY_DIR / "registry.json").read_text())
    conditions = list(registry["conditions"])

    for condition in conditions:
        for domain in ["tennis", "tiser"]:
            !python scripts/experiment.py evaluate \
                --study-dir "{STUDY_DIR}" \
                --condition "{condition}" \
                --domain "{domain}" \
                --stage selection \
                --audit-dir "{AUDIT_DIR}" \
                --resume

    !python scripts/experiment.py freeze-final \
        --study-dir "{STUDY_DIR}" \
        --audit-dir "{AUDIT_DIR}"

campaign = json.loads(campaign_path.read_text())

for condition in campaign["conditions"]:
    completion_path = (
        STUDY_DIR / "evaluations/final/tennis" / condition / "completion.json"
    )
    completion = (
        json.loads(completion_path.read_text()) if completion_path.exists() else {}
    )

    if completion.get("status") == "complete":
        print("Using completed final tennis evaluation:", condition)
    else:
        !python scripts/experiment.py evaluate \
            --study-dir "{STUDY_DIR}" \
            --condition "{condition}" \
            --domain tennis \
            --stage final \
            --audit-dir "{AUDIT_DIR}" \
            --resume


Using completed final tennis evaluation: C0
Using completed final tennis evaluation: C1


## Final paired tennis statistics

Compute uncertainty and the paired significance test from the completed predictions.


In [ ]:
statistics_path = STUDY_DIR / "final_tennis_statistics/statistics.json"

if not statistics_path.exists():
    !python scripts/experiment.py statistics \
        --study-dir "{STUDY_DIR}"

statistics = json.loads(statistics_path.read_text())
print(json.dumps(statistics, indent=2))


## Reporting
Use audited primary tennis scores with their actual eligible denominator and
show original-label scores separately. Original-TISER retention is the unweighted
five-split macro; ToT-semantic is secondary OOD evidence. Preserve bootstrap
replicates and Holm-adjusted per-split McNemar tables. A single training seed
does not measure training variability. Model-judge agreement does not establish
human-calibrated accuracy. The holdout prior-use statement remains qualified.
